In [ ]:
import sys
import os
from dotenv import load_dotenv
sys.path.append(os.path.abspath("../.."))
from src.config.llm import GoogleLLm
from langchain_google_genai import ChatGoogleGenerativeAI

from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from pydantic import BaseModel,Field
load_dotenv()

True

In [9]:


model = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')





In [24]:
class SentimentSchema(BaseModel):
    sentiment:Literal['positive','negative'] = Field(description="Sentiment of the review")

class ReviewState(TypedDict):
    review:str
    sentiment:Literal['positive','negative']
    diagnosis:dict
    response:str

class DiagnosisSchema(BaseModel):
     issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
     tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
     urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')



In [25]:
final_model = model.with_structured_output(SentimentSchema)
dianosis_model = model.with_structured_output(DiagnosisSchema)


In [26]:
def find_sentiment(state:ReviewState):
    review = state['review']
    prompt = f"""for the following review find out the sentiment \n {review}"""
    sentiment =final_model.invoke(prompt).sentiment
    return{'sentiment':sentiment}

def check_sentiment(state:ReviewState)->Literal["positive_response","run_diagnosis"]:
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'
    
def positive_response(state:ReviewState):
    prompt = f"Write a warm thankyou message in the response of the review: \n {state['review']}\n also kindly ask the user to leave a feedback  on our website"
    response = model.invoke(prompt)
    return{'response':response}


def run_diagnosis(state:ReviewState):
    prompt =f"""diagnose this negative review:\n\n{state['review']}\n
           Return issue_type, tone and urgency
"""
    response = dianosis_model.invoke(prompt)
    return {'diagnosis':response.model_dump()}

def negative_response(state:ReviewState):
    diagnosis = state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message.
"""
    response = model.invoke(prompt).content
    return {'response':response}

In [27]:
#graph initlize
graph = StateGraph(ReviewState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_response",positive_response)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("negative_response",negative_response)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",check_sentiment)
graph.add_edge("positive_response",END)
graph.add_edge("run_diagnosis","negative_response")
graph.add_edge("negative_response",END)

In [29]:
#complie the graph
workflow = graph.compile()
workflow

initial_state={
    'review': "I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(initial_state)

{'review': 'I’ve been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.',
 'sentiment': 'negative',
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': "I'm so sorry to hear you're experiencing a bug that's causing you frustration, especially with high urgency! I completely understand how disruptive and upsetting that can be.\n\nPlease know that we're taking this very seriously and are committed to getting this resolved for you as quickly as possible.\n\nTo help us pinpoint the issue and find the best solution, could you please provide us with a little more detail? Specifically, any information you can share about:\n\n*   **What exactly is happening?** (e.g., error messages, unexpected behavior, specific features not working)\n*   **When did this bug start occurring?**\n*   **Are